# 45. 중심을 고정하고 원판 박스 크기만 보정하면 IoU가 오를까?

44번과 같은 여섯 선수의 무릎 통과→접촉 56장을 사용합니다. **주 평가는 원본 56장**입니다. 채도 0.5배에서 IoU가 0.4993이었던 b02 33번은 별도 진단 사례입니다. 1.5배도 같은 규칙으로 확인합니다.

현재 예측 중심을 그대로 두고, 너비와 높이는 각각 **(현재 값 + 직전 원본 값) / 2**로 계산합니다. 매번 보정 전 박스를 사용합니다. 현재나 직전 박스가 없거나 보정 후 화면 경계를 넘으면 기존 출력을 유지합니다. 후보 선택과 다음 프레임의 이력에 보정값을 전달하지 않습니다. 세 조건 모두 같은 원본 이력을 사용합니다.

## 1. 로컬 비교 자료 확인

VS Code에서 프로젝트의 Python 커널을 선택하고 위에서 아래로 실행하세요. 40번에서 계산한 전체 영상의 원판 선택 표와 44번의 프레임별 결과를 읽습니다. 여러 회차가 있으면 아래 두 입력 경로를 확인하세요. 선택 표의 원본 박스와 점수가 44번과 일치하는지 검사합니다.

In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import sys

start = Path(globals().get("__vsc_ipynb_file__", Path.cwd())).resolve()
root = next(p for p in (start, *start.parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from wlcv.evaluation.box_size_review import compare_sizes, plot_example, summarize

outputs = root / "outputs"
def latest_input(pattern):
    files = list(outputs.glob(pattern))
    if not files:
        raise FileNotFoundError(f"{pattern} 파일이 필요합니다. 40번·44번의 결과 저장 셀을 확인하세요.")
    return max(files, key=lambda file: file.stat().st_mtime)

frame_file = latest_input("plate_saturation_review/*/frames.csv")
history_file = latest_input("phase_speed/*/selected.csv")
settings = json.loads((frame_file.parent / "settings.json").read_text())
assert (settings["imgsz"], settings["direct_conf"], settings["link_radius_ratio"], settings["match_iou"]) == (640, .1, .5, .5)
frames = pd.read_csv(frame_file)
history = pd.read_csv(history_file)
metadata = pd.read_csv(root / "data/metadata/videos.csv")
print(frame_file, history_file, sep="\n")
display(frames[frames.factor.eq(1)].groupby("person_id").frame_index.agg(["min", "max", "count"]))

## 2. 56장 모두에서 정탐·IoU·아랫변 오차 비교

세 조건은 같은 프레임을 반복 사용합니다. 각각 56장씩 따로 해석하세요. IoU 0.50 이상이면 정탐입니다. 복구와 손실은 그 경계를 새로 넘거나 벗어난 장수입니다. 평균 IoU와 아랫변 평균 절대오차(px)는 박스가 출력된 장면에서 계산합니다.

In [ ]:
results = compare_sizes(frames, history, metadata)
summary = summarize(results, ["factor"])
people = summarize(results, ["factor", "person_id"])
display(summary.round(4))
display(people[people.factor.eq(1)].round(4))
print("최대 중심 이동량 (px)", results[["shift_x", "shift_y"]].abs().max().to_dict())
changes = results[results.recovered | results.lost]
display(changes[["person_id", "frame_index", "factor", "source", "iou_before", "iou_after", "recovered", "lost"]].round(4))

## 3. b02 33번과 선수별 결과 보기

세 칸 모두 같은 원본 RGB를 배경으로 두고 각 채도 조건에서 선택한 박스를 그립니다. 흰 점선은 수동 박스, 주황 파선은 기존 예측, 파란 실선은 보정 후 예측입니다. 십자는 두 예측이 공유하는 중심입니다. 그래프는 원본 영상에서 선수별 평균 IoU 변화를 보여줍니다.

In [ ]:
example = plot_example(results)
plt.show()
original = people[people.factor.eq(1)].set_index("person_id")
delta = original.mean_iou_after - original.mean_iou_before
fig, ax = plt.subplots(figsize=(7, 3.2), layout="constrained")
ax.bar(delta.index, delta.values, color=["#287e83" if d >= 0 else "#b75e43" for d in delta])
ax.axhline(0, color="black", lw=0.8)
ax.set(xlabel="Validation athlete", ylabel="Mean IoU change (after − before)", title="Original frames")
plt.show()

## 4. 실행 결과 저장

원본에서 선수별 IoU 변화 방향, 복구한 프레임과 잃은 프레임, 아랫변 오차를 함께 보세요. 중심 좌표가 같으면 같은 프레임에서의 허브 세로 오차·상승량·속도도 같습니다. IoU 통과 장수의 변화는 따로 기록합니다.

In [ ]:
folder = outputs / "plate_box_size_review" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
folder.mkdir(parents=True)
results.to_csv(folder / "frames.csv", index=False)
summary.to_csv(folder / "summary.csv", index=False)
people.to_csv(folder / "people.csv", index=False)
example.savefig(folder / "b02_frame33.png", dpi=150)
fig.savefig(folder / "original_iou_by_person.png", dpi=150)
run = dict(current_weight=0.5, previous_weight=0.5, previous="raw original t-1", match_iou=0.5,
           weights_sha256=settings["weights_sha256"], inputs={str(p.relative_to(root)): hashlib.sha256(p.read_bytes()).hexdigest()
                                                             for p in (frame_file, history_file)})
(folder / "settings.json").write_text(json.dumps(run, indent=2))
print(folder)